<a href="https://colab.research.google.com/github/tettehdaniel/python-labs/blob/main/Notebooks/Notebook05_DataManipulation3_SV.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Notebook 05 — Data Manipulation Three

**Main Objective**

1. Aggregation/Summarizing
2. Piping

## Setup

Please run the setup code below

In [1]:
! wget -q -nc https://raw.githubusercontent.com/taylor-arnold/fds-py/refs/heads/main/funs.py

from funs import *

In [2]:
import numpy as np
import polars as pl

from plotnine import *
from polars import col as c
theme_set(theme_minimal())

path = "https://raw.githubusercontent.com/ajcollins16/DSST289_Fall2026_AJC/refs/heads/main/"

In [41]:
dat = pl.read_csv(path + "Data/criterion.csv").filter(~c.rating_imdb.is_null())

## Summarizing and Piping

Another day another set of new data manipulation techniques to work through. Specifically, we will work through the process of summarizing data with the `.agg` function and broadly discussing the `.pipe` function. What we mean by summarizing is that many times we want to get some descriptive statistics (i.e., mean, median, mode, variance, standard deviation, quantiles, etc.) or other meaningful pieces of information (i.e., max, min, sample size, correlation, etc.) on a variable of interest out.

Today's dataset is a film dataset that contains information about movies like title, director, release year, runtime, and the IMDB rating (at the time).

In [4]:
dat

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0012349""","""The Kid""",1921,"""None|English""","""Comedy|Drama|Family""","""Charles Chaplin""","""Charles Chaplin""","""United States""",142797,8.2,100,null,68,1346905,"""1921 silent film by Charlie Ch…","""The Kid is a 1921 American sil…",250000,null
"""tt0012364""","""The Phantom Carriage""",1921,"""None|Swedish""","""Drama|Fantasy|Horror""","""Victor Sjöström""","""Selma Lagerlöf|Victor Sjöström""","""Sweden""",15311,8.0,100,null,106,7329426,"""1921 film by Victor Sjöström""","""The Phantom Carriage is a 1921…",null,null
"""tt0013257""","""Häxan""",1922,"""Swedish|Danish""","""Documentary|Fantasy|Horror""","""Benjamin Christensen""","""Benjamin Christensen""","""Sweden|Denmark""",18391,7.6,93,null,107,3644898,"""Swedish 1922 silent horror ess…","""Häxan is a 1922 Swedish-Danish…",2000000,null
"""tt0014429""","""Safety Last!""",1923,"""English""","""Action|Comedy|Thriller""","""Fred C. Newmeyer|Sam Taylor""","""Hal Roach|Sam Taylor|Tim Whela…","""United States""",23503,8.1,97,null,73,76313,"""1923 American silent romantic …","""Safety Last! is a 1923 America…",121000,null
"""tt0014624""","""A Woman of Paris: A Drama of F…",1923,"""English""","""Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",6548,6.9,94,76,84,546663,"""1923 drama film by Charlie Cha…","""A Woman of Paris is a 1923 sil…",351000,634000
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt27958252""","""Caught by the Tides""",2024,"""Mandarin|English""","""Drama""","""Jia Zhang-ke""","""Jiahuan Wan|Jia Zhang-ke""","""China|France|Japan""",2097,6.7,99,87,111,76585127,"""2024 film by Jia Zhangke""","""Caught by the Tides is a 2024 …",null,null
"""tt28607951""","""Anora""",2024,"""English|Russian|Armenian""","""Comedy|Drama|Romance""","""Sean Baker""","""Sean Baker""","""United States""",235046,7.4,93,91,139,75146745,"""2024 film by Sean Baker""","""Anora is a 2024 American roman…",null,null
"""tt28618488""","""Vermiglio""",2024,"""Italian|Latin""","""Drama|History""","""Maura Delpero""","""Maura Delpero""","""Italy|France|Belgium""",4838,6.9,93,85,119,77421920,"""2024 film by Maura Delpero""","""Vermiglio , also known as The …",null,null


### Questions

1. Start by sorting the films by their IMDB rating, with the lowest-rated film at the top.

In [5]:
(
    dat
    .sort(c.rating_imdb)
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0062492""","""Wild 90""",1968,"""English""","""Comedy|Crime|Drama""","""Norman Mailer""","""D.A. Pennebaker""","""United States""",139,3.5,null,null,90,20765561,"""1968 film by Norman Mailer""","""Wild 90 is a 1968 experimental…",null,null
"""tt0064373""","""All Monsters Attack""",1969,"""Japanese""","""Adventure|Family|Fantasy""","""Ishirô Honda|Jun Fukuda|Kengo …","""Shin'ichi Sekizawa""","""Japan|United States""",5763,3.9,38,null,69,570240,"""1969 film by Ishirō Honda""","""All Monsters Attack is a 1969…",null,null
"""tt0063195""","""Genocide""",1968,"""Japanese""","""Horror|Sci-Fi""","""Kazui Nihonmatsu""","""Kingen Amada|Susumu Takaku""","""Japan""",802,4.7,null,null,84,51448466,"""1968 Japanese film""","""Genocide is a 1968 Japanese s…",null,null
"""tt0064625""","""Maidstone""",1970,"""English""","""Drama""","""Norman Mailer""","""Norman Mailer""","""United States""",280,4.7,null,null,null,4586481,"""1970 film by Norman Mailer""","""Maidstone is a 1970 American i…",null,null
"""tt2006181""","""Jellyfish Eyes""",2013,"""Japanese""","""Comedy|Fantasy""","""Takashi Murakami""","""Takashi Murakami|Yoshihiro Nis…","""Japan""",574,4.7,29,34,101,41633681,"""2013 Japanese film""","""Jellyfish Eyes is a 2013 Japa…",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt0048434""","""Night and Fog""",1956,"""French""","""Documentary|Short|History""","""Alain Resnais""","""Jean Cayrol""","""France""",22148,8.6,null,null,32,20181860,"""1956 French film""","""Night and Fog is a 1956 French…",null,null
"""tt0056058""","""Harakiri""",1962,"""Japanese""","""Drama|Mystery""","""Masaki Kobayashi""","""Yasuhiko Takiguchi|Shinobu Has…","""Japan""",87601,8.6,100,85,135,3137571,"""Japanese jidaigeki film""","""Harakiri is a 1962 Japanese j…",null,15000
"""tt0102926""","""The Silence of the Lambs""",1991,"""English|Latin""","""Crime|Drama|Horror""","""Jonathan Demme""","""Thomas Harris|Ted Tally""","""United States""",1667162,8.6,95,86,118,30006,"""1991 horror film by Jonathan D…","""The Silence of the Lambs is a …",19000000,130742922


2. Now, sort the films by runtime with the longest films at the top. Note, use the `.drop_nulls` function to make sure you don't have movies at the top with no runtime given.

In [31]:
(
    dat
    .drop_nulls(c.runtime_raw)
    .sort(c.runtime_raw, descending = True)
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0090015""","""Shoah""",1985,"""German|Hebrew|Polish|Yiddish|F…","""Documentary|History|War""","""Claude Lanzmann""","""Claude Lanzmann""","""France""",11202,8.7,100,99,540,147213,"""1985 French documentary film b…","""Shoah is a 1985 French documen…",null,null
"""tt0063794""","""War and Peace""",1965,"""Russian|German|French""","""Drama|Romance|War""","""Sergey Bondarchuk""","""Sergey Bondarchuk|Vasiliy Solo…","""Soviet Union""",8619,8.3,100,null,432,2792417,"""1967 film by Sergei Bondarchuk""","""War and Peace is a 1965–1967 S…",8291712,58000000
"""tt0025509""","""Les Misérables""",1934,"""French""","""Drama""","""Raymond Bernard""","""Victor Hugo|André Lang|Raymond…","""France""",2084,8.3,null,null,281,2126621,"""1934 French film""","""Les Misérables is a 1934 Frenc…",null,null
"""tt0101985""","""A Brighter Summer Day""",1991,"""Mandarin|Min Nan|Shanghainese|…","""Crime|Drama|Romance""","""Edward Yang""","""Edward Yang|Alex Yang|Mingtang…","""Taiwan""",14580,8.2,100,91,237,7595441,"""1991 Taiwanese epic coming-of-…","""A Brighter Summer Day is a 199…",null,null
"""tt0070359""","""The Mother and the Whore""",1973,"""French""","""Drama|Romance""","""Jean Eustache""","""Jean Eustache""","""France""",7217,7.7,94,89,217,10809839,"""1973 French film""","""The Mother and the Whore is a …",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt0056662""","""Vive le tour""",1962,"""French""","""Documentary|Short|Sport""","""Louis Malle""","""Louis Malle""","""France""",999,7.4,null,null,18,22670309,"""1962 French film""","""Vive le Tour is a 1962 French …",null,null
"""tt0075122""","""The Colours""",1976,"""Persian""","""Documentary|Short""","""Abbas Kiarostami""","""Abbas Kiarostami""","""Iran""",397,6.2,null,null,15,9308455,"""1976 Iranian film""","""The Colours is a 1976 Iranian …",null,null
"""tt0063551""","""Saute ma ville""",1971,"""French""","""Short|Comedy|Drama""","""Chantal Akerman""","""N/A""","""Belgium""",1283,6.3,null,null,13,308673,"""Belgian film director, screenw…","""Chantal Anne Akerman was a Bel…",null,null


3. Now select only the films that are less than 60 minutes long (i.e., short films).

In [32]:
(
    dat
    .filter(c.runtime_raw < 60)
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0021576""","""À Propos de Nice""",1930,"""None""","""Documentary|Short|Comedy""","""Boris Kaufman|Jean Vigo""","""Jean Vigo|Boris Kaufman""","""France""",4591,7.4,null,null,25,11286391,"""1930 film""","""À propos de Nice is a 1930 sil…",null,null
"""tt0022036""","""Flunky, Work Hard!""",1931,"""None|Japanese""","""Short|Comedy|Drama""","""Mikio Naruse""","""Mikio Naruse""","""Japan""",531,6.7,null,null,29,66536197,"""1931 Japanese film""","""Flunky, Work Hard! is a 1931 …",null,null
"""tt0024803""","""Zero for Conduct""",1933,"""French""","""Drama""","""Jean Vigo""","""Jean Vigo""","""France""",9513,7.2,94,null,44,38244256,"""1933 French film""","""Zero for Conduct is a 1933 Fre…",null,null
"""tt0028445""","""A Day in the Country""",1936,"""French""","""Short|Comedy|Drama""","""Jean Renoir""","""Guy de Maupassant|Jean Renoir""","""France""",7611,7.5,100,null,40,7134551,"""1946 French film""","""Partie de campagne is a French…",null,null
"""tt0038182""","""The Men Who Tread on the Tiger…",1945,"""Japanese""","""Adventure|Drama|Thriller""","""Akira Kurosawa""","""Nobumitsu Kanze|Akira Kurosawa…","""Japan""",4721,6.7,86,null,59,1529461,"""1945 Japanese film by Akira Ku…","""The Men Who Tread on the Tiger…",null,null
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt0079704""","""Paul Robeson: Tribute to an Ar…",1979,"""English|Mandarin""","""Documentary|Short|Biography""","""Saul J. Turell""","""Saul J. Turell""","""United States""",521,7.5,null,null,30,17656522,"""1979 film""","""Paul Robeson: Tribute to an Ar…",null,null
"""tt0081204""","""Mur murs""",1981,"""French|English|Italian|Spanish""","""Documentary""","""Agnès Varda""","""Agnès Varda""","""France|United States|West Germ…",1269,7.4,null,null,58,48394918,"""1980 documentary film by Agnès…","""Mur Murs is a 1981 documentary…",null,null
"""tt0083727""","""Chambre 666""",1982,"""English|French|German|Italian|…","""Documentary""","""Wim Wenders""","""Wim Wenders""","""France|West Germany""",1877,6.6,null,null,45,11105410,"""1982 French film""","""Room 666 is a 1982 documentary…",null,null


4. Now select the films that are directed by Charles Chaplin.

In [33]:
(
    dat
    .filter(c.director == "Charles Chaplin")
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0012349""","""The Kid""",1921,"""None|English""","""Comedy|Drama|Family""","""Charles Chaplin""","""Charles Chaplin""","""United States""",142797,8.2,100,null,68,1346905,"""1921 silent film by Charlie Ch…","""The Kid is a 1921 American sil…",250000,null
"""tt0014624""","""A Woman of Paris: A Drama of F…",1923,"""English""","""Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",6548,6.9,94,76,84,546663,"""1923 drama film by Charlie Cha…","""A Woman of Paris is a 1923 sil…",351000,634000
"""tt0018773""","""The Circus""",1928,"""None|English""","""Comedy|Family|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",37901,8.1,97,90,69,142780,"""1928 film by Charlie Chaplin""","""The Circus is a 1928 silent ro…",null,null
"""tt0021749""","""City Lights""",1931,"""None|English""","""Comedy|Drama|Romance""","""Charles Chaplin""","""Charles Chaplin|Harry Carr|Har…","""United States""",210306,8.5,95,99,87,73401,"""1931 American silent film""","""City Lights is a 1931 American…",1500000,5000000
"""tt0027977""","""Modern Times""",1936,"""English""","""Comedy|Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",276967,8.5,98,96,99,74962,"""1936 comedy film by Charlie Ch…","""Modern Times is a 1936 America…",1500000,1400000
"""tt0032553""","""The Great Dictator""",1940,"""English|Esperanto|Latin""","""Comedy|Drama|War""","""Charles Chaplin""","""Charles Chaplin""","""United States""",252228,8.4,92,null,125,62689,"""1940 American film by Charlie …","""The Great Dictator is a 1940 A…",2000000,5000000
"""tt0039631""","""Monsieur Verdoux""",1947,"""English|French|Latin""","""Comedy|Crime|Drama""","""Charles Chaplin""","""Charles Chaplin|Orson Welles|M…","""United States""",19567,7.8,97,null,124,545615,"""1947 film by Charlie Chaplin""","""Monsieur Verdoux is a 1947 Ame…",2000000,null
"""tt0044837""","""Limelight""",1952,"""English""","""Comedy|Drama|Music""","""Charles Chaplin""","""Charles Chaplin""","""United States""",23456,8.0,88,84,137,583273,"""1952 film by Charlie Chaplin""","""Limelight is a 1952 American c…",900000,1000000


5. Now select the films direct by either Charles Chaplin or Jean-Luc Godard.

In [35]:
(
    dat
    .filter((c.director == "Charles Chaplin") | (c.director == "Jean-Luc Godard"))
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64
"""tt0012349""","""The Kid""",1921,"""None|English""","""Comedy|Drama|Family""","""Charles Chaplin""","""Charles Chaplin""","""United States""",142797,8.2,100,null,68,1346905,"""1921 silent film by Charlie Ch…","""The Kid is a 1921 American sil…",250000,null
"""tt0014624""","""A Woman of Paris: A Drama of F…",1923,"""English""","""Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",6548,6.9,94,76,84,546663,"""1923 drama film by Charlie Cha…","""A Woman of Paris is a 1923 sil…",351000,634000
"""tt0018773""","""The Circus""",1928,"""None|English""","""Comedy|Family|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",37901,8.1,97,90,69,142780,"""1928 film by Charlie Chaplin""","""The Circus is a 1928 silent ro…",null,null
"""tt0021749""","""City Lights""",1931,"""None|English""","""Comedy|Drama|Romance""","""Charles Chaplin""","""Charles Chaplin|Harry Carr|Har…","""United States""",210306,8.5,95,99,87,73401,"""1931 American silent film""","""City Lights is a 1931 American…",1500000,5000000
"""tt0027977""","""Modern Times""",1936,"""English""","""Comedy|Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",276967,8.5,98,96,99,74962,"""1936 comedy film by Charlie Ch…","""Modern Times is a 1936 America…",1500000,1400000
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt0060675""","""Masculine Feminine""",1966,"""French|Swedish|English""","""Drama|Romance""","""Jean-Luc Godard""","""Jean-Luc Godard|Guy de Maupass…","""France|Sweden""",18464,7.4,96,93,104,1644429,"""1966 film by Jean-Luc Godard""","""Masculin féminin: 15 Specific …",null,null
"""tt0060304""","""2 or 3 Things I Know About Her""",1967,"""French|Italian|English""","""Comedy|Drama""","""Jean-Luc Godard""","""Catherine Vimenet|Jean-Luc God…","""France""",8772,6.5,94,null,90,1608743,"""1967 film by Jean-Luc Godard""","""Two or Three Things I Know Abo…",null,null
"""tt0062480""","""Weekend""",1967,"""French""","""Adventure|Comedy|Drama""","""Jean-Luc Godard""","""Julio Cortázar|Jean-Luc Godard""","""France|Italy""",16133,6.9,93,null,103,2117053,"""Film by Jean-Luc Godard""","""Weekend is a 1967 postmodern b…",null,null


6. For each director, compute the average IMDB ratings of their films and the number of films they have in their collection.

In [15]:
(
    dat
    .group_by(c.director)
    .agg(
        num_movie = pl.len(),
        mean_imdb = c.rating_imdb.mean()
    )
)

director,num_movie,mean_imdb
str,u32,f64
"""Michael Powell""",2,7.45
"""Idrissa Ouedraogo""",1,6.5
"""Ted Wilde""",1,7.6
"""Donna Deitch""",1,7.1
"""William Dieterle""",1,7.6
…,…,…
"""Ermanno Olmi""",3,7.766667
"""Tokuzô Tanaka""",3,7.233333
"""Bruce Lee""",1,7.1


One interesting thing that is often done with movies that we can practice here is to group movies by decade rather than individual years. To accomplish this we can use the expression `c.year // 10 * 10`. The `//` operator in Python performans something called **integer division**, which divides the numbers and then drops the decimals/remainder.

7. As an example can you integer divide 1987 from 10 below

In [18]:
1987 // 10 * 10

1980

7. What do you get as output?

**Answer (Q7):**

8. Create a new column in our dataset containing the decade each film was released.

In [19]:
(
    dat
    .with_columns(decade = c.year // 10 * 10)
)

imdb_id,title,year,language,genre,director,writer,country,imdb_votes,rating_imdb,rating_rt,rating_mc,runtime_raw,wikipedia_pageid,wikipedia_description,wikipedia_extract,budget_raw,box_office_raw,decade
str,str,i64,str,str,str,str,str,i64,f64,i64,i64,i64,i64,str,str,i64,i64,i64
"""tt0012349""","""The Kid""",1921,"""None|English""","""Comedy|Drama|Family""","""Charles Chaplin""","""Charles Chaplin""","""United States""",142797,8.2,100,null,68,1346905,"""1921 silent film by Charlie Ch…","""The Kid is a 1921 American sil…",250000,null,1920
"""tt0012364""","""The Phantom Carriage""",1921,"""None|Swedish""","""Drama|Fantasy|Horror""","""Victor Sjöström""","""Selma Lagerlöf|Victor Sjöström""","""Sweden""",15311,8.0,100,null,106,7329426,"""1921 film by Victor Sjöström""","""The Phantom Carriage is a 1921…",null,null,1920
"""tt0013257""","""Häxan""",1922,"""Swedish|Danish""","""Documentary|Fantasy|Horror""","""Benjamin Christensen""","""Benjamin Christensen""","""Sweden|Denmark""",18391,7.6,93,null,107,3644898,"""Swedish 1922 silent horror ess…","""Häxan is a 1922 Swedish-Danish…",2000000,null,1920
"""tt0014429""","""Safety Last!""",1923,"""English""","""Action|Comedy|Thriller""","""Fred C. Newmeyer|Sam Taylor""","""Hal Roach|Sam Taylor|Tim Whela…","""United States""",23503,8.1,97,null,73,76313,"""1923 American silent romantic …","""Safety Last! is a 1923 America…",121000,null,1920
"""tt0014624""","""A Woman of Paris: A Drama of F…",1923,"""English""","""Drama|Romance""","""Charles Chaplin""","""Charles Chaplin""","""United States""",6548,6.9,94,76,84,546663,"""1923 drama film by Charlie Cha…","""A Woman of Paris is a 1923 sil…",351000,634000,1920
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""tt27958252""","""Caught by the Tides""",2024,"""Mandarin|English""","""Drama""","""Jia Zhang-ke""","""Jiahuan Wan|Jia Zhang-ke""","""China|France|Japan""",2097,6.7,99,87,111,76585127,"""2024 film by Jia Zhangke""","""Caught by the Tides is a 2024 …",null,null,2020
"""tt28607951""","""Anora""",2024,"""English|Russian|Armenian""","""Comedy|Drama|Romance""","""Sean Baker""","""Sean Baker""","""United States""",235046,7.4,93,91,139,75146745,"""2024 film by Sean Baker""","""Anora is a 2024 American roman…",null,null,2020
"""tt28618488""","""Vermiglio""",2024,"""Italian|Latin""","""Drama|History""","""Maura Delpero""","""Maura Delpero""","""Italy|France|Belgium""",4838,6.9,93,85,119,77421920,"""2024 film by Maura Delpero""","""Vermiglio , also known as The …",null,null,2020


9. Now, find the count/sample size for how many films in the collection were released in each decade.

In [40]:
(
    dat
    .with_columns(decade = c.year // 10 * 10)
    .group_by(c.decade)
    .agg(
        num_movie = pl.len()
    )
)

decade,num_movie
i64,u32
1920,19
1960,303
1970,232
1980,171
1990,154
…,…
2020,47
1930,119
1940,122


10. Find the directors with more than 5 films in their collection, showing their average IMDB rating. Sort the results so the highest-rated directors appear at the top.

In [28]:
(
    dat
    .sort(c.rating_imdb, descending = True)
    .group_by(c.director)
    .agg(
        num_movie = pl.len(),
        avg_rating = c.rating_imdb.mean()
    )
    .filter(c.num_movie > 5)
)

director,num_movie,avg_rating
str,u32,f64
"""Akira Kurosawa""",24,7.525
"""Michelangelo Antonioni""",7,7.414286
"""Michael Powell|Emeric Pressbur…",7,7.657143
"""Ernst Lubitsch""",8,7.375
"""Kenji Mizoguchi""",9,7.777778
…,…,…
"""Alfred Hitchcock""",7,7.528571
"""Yasujirô Ozu""",20,7.62
"""Terry Gilliam""",6,7.15


11. Continuing, for each director with more than 5 films in their collection, show their highest-rated film. Hint: If you sort by rating before grouping, the first title in each group will be that director's highest-rated film.

In [25]:
(
    dat
    .sort(c.rating_imdb, descending = True)
    .group_by(c.director)
    .agg(
        num_movie = pl.len(),
        highest_imdb = c.title.first()
    )
    .filter(c.num_movie > 5)
)

director,num_movie,highest_imdb
str,u32,str
"""Louis Malle""",13,"""Au Revoir les Enfants"""
"""Michael Powell|Emeric Pressbur…",7,"""The Red Shoes"""
"""Michael Haneke""",6,"""The Seventh Continent"""
"""Keisuke Kinoshita""",7,"""Twenty-Four Eyes"""
"""John Cassavetes""",7,"""A Woman Under the Influence"""
…,…,…
"""Alfred Hitchcock""",7,"""Rebecca"""
"""David Cronenberg""",7,"""A History of Violence"""
"""Carl Theodor Dreyer""",6,"""Ordet"""


12. By default, only the first few rows of a result are displayed. To see more rows, we can pipe the result through the `print_rows` function. Modify your previous answer to display all of the results.

In [26]:
(
    dat
    .sort(c.rating_imdb, descending = True)
    .group_by(c.director)
    .agg(
        num_movie = pl.len(),
        highest_imdb = c.title.first()
    )
    .filter(c.num_movie > 5)
    .pipe(print_rows)
)

director,num_movie,highest_imdb
str,u32,str
"""Samuel Fuller""",7,"""Pickup on South Street"""
"""Masaki Kobayashi""",7,"""Harakiri"""
"""Terry Gilliam""",6,"""Brazil"""
"""Michael Haneke""",6,"""The Seventh Continent"""
"""David Lynch""",6,"""Mulholland Drive"""
"""Wong Kar-Wai""",7,"""In the Mood for Love"""
"""Yasujirô Ozu""",20,"""Late Spring"""
"""Michael Powell|Emeric Pressbur…",7,"""The Red Shoes"""
"""Martin Scorsese""",8,"""The Last Waltz"""
